# NOMAD Schema Tutorial — Planck's Blackbody Radiation

This notebook demonstrates how a NOMAD schema works end-to-end:

1. **Define** quantities, sub-sections, and a `normalize` method in a schema class
2. **Run** the normalizer to compute derived data from user inputs
3. **Inspect** the populated archive — scalar results, arrays, and `archive.results`
4. **Visualise** the stored Plotly figure

The example schema (`BlackbodyRadiation`) takes a single temperature value as input and computes the full spectral radiance profile B(λ, T) using Planck's law:

$$B(\lambda, T) = \frac{2hc^2}{\lambda^5} \cdot \frac{1}{e^{hc / \lambda k_B T} - 1}$$

The peak emission wavelength is derived via Wien's displacement law and stored in `archive.results` for searchability.

## Step 1: Create an archive and populate the schema

An `EntryArchive` is the top-level NOMAD data container. We attach a `BlackbodyRadiation` instance to `archive.data` — this is what a user would fill in via the ELN form.

Calling `normalize_all` runs all `normalize` methods defined in the schema, which computes the spectrum and populates the results.

In [1]:
from nomad.client import normalize_all
from nomad.datamodel.datamodel import EntryArchive

# Let's build a dummy NOMAD archive that mimicks a real NOMAD entry archive that can be
# created by a user in the GUI.
archive = EntryArchive(
    metadata={
        'entry_name': 'blackbody_test',
        'entry_type': 'BlackbodyRadiation',
    },
)

/home/sarthak-kapoor/repositories/fairmat/dev_distro/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING  MDAnalysis.coordinat 2026-06-08T13:22:37Z netCDF4 is not available. Writing AMBER ncdf files will be slow.
  - nomad.commit: 
  - nomad.deployment: oasis
  - nomad.service: unknown nomad service
  - nomad.version: 1.4.3.dev134+g2ebf1e4f7
  - taskName: Task-24


In [11]:
from nomad_parser_tutorial_exercise.schema_tutorial.schema_package import (
    BlackbodyRadiation,
)

archive.data = BlackbodyRadiation(
    name='Molten Iron',
    temperature=1800.0,
)
normalize_all(archive)

print(
    'Peak Wavelength @ `archive.data.results`: '
    f'{archive.data.results.peak_wavelength}'
)
print(
    'Material Name @ `archive.results.material`: '
    f'{archive.results.material.material_name}'
)

Peak Wavelength @ `archive.data.results`: 1609.8733083333332 nanometer
Material Name @ `archive.results.material`: Molten Iron


In [12]:
import plotly.graph_objects as go

print(
    'Custom Plotly figure added by the data normalizer'
    ' @ `archive.data.results.figures[0]`:'
)
fig = go.Figure(archive.data.results.figures[0].figure)
fig.update_layout(width=600, height=400)
fig.show()

Custom Plotly figure added by the data normalizer @ `archive.data.results.figures[0]`:
